In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os

# --- CONFIGURATION ---
input_file = '../Results/final_radiomics_merged.csv' 
output_file = '../Results/table_statistical_analysis.csv'

print("--- AUTOMATED STATISTICAL AUDIT ---")

if not os.path.exists(input_file):
    print(f"ERROR: File not found: {input_file}")
else:
    df = pd.read_csv(input_file)
    print(f"Data Loaded: {len(df)} patients")
    
    cohorts = df['Cohort'].unique()
    print(f"Cohorts found: {cohorts}")
    
    if len(cohorts) < 2:
        print("Error: Need at least 2 cohorts to compare.")
    else:
        group1_name = cohorts[0]
        group2_name = cohorts[1]
        print(f"\nComparing: '{group1_name}' vs. '{group2_name}'")

        group1 = df[df['Cohort'] == group1_name]
        group2 = df[df['Cohort'] == group2_name]

        features = [c for c in df.columns if 'original_' in c]
        print(f"Analyzing {len(features)} radiomic features...")

        results = []

        for feature in features:
            v1 = group1[feature].dropna()
            v2 = group2[feature].dropna()
            
            # Descriptive Stats
            mean1, sd1 = v1.mean(), v1.std()
            mean2, sd2 = v2.mean(), v2.std()
            
            # Normality Check
            try:
                if len(v1) > 3 and len(v2) > 3:
                    _, p_norm1 = stats.shapiro(v1)
                    _, p_norm2 = stats.shapiro(v2)
                    is_normal = (p_norm1 > 0.05) and (p_norm2 > 0.05)
                else:
                    is_normal = False
            except:
                is_normal = False 

            # Significance Testing
            p_value = 1.0
            test_used = ""
            
            if is_normal:
                statistic, p_value = stats.ttest_ind(v1, v2, equal_var=False)
                test_used = "T-Test (Welch)"
            else:
                statistic, p_value = stats.mannwhitneyu(v1, v2)
                test_used = "Mann-Whitney U"
                
            # Levene's Test
            try:
                lev_stat, lev_p = stats.levene(v1, v2)
            except:
                lev_p = 1.0

            # Fold Change
            fold_change = mean2 / (mean1 + 1e-9)

            # Append Result - FIXED COLUMN NAME KEYS
            results.append({
                'Feature': feature,
                f'Mean ({group1_name})': f"{mean1:.2f} ± {sd1:.2f}",
                f'Mean ({group2_name})': f"{mean2:.2f} ± {sd2:.2f}",
                'Fold Change': f"{fold_change:.2f}x",
                'P_Value': p_value,  # Changed from 'P-Value' to 'P_Value' to match sort
                'Test Used': test_used,
                'Levene Variance P-Val': lev_p,
                'Significant?': 'YES' if p_value < 0.05 else 'No'
            })

        # Save & Display
        df_results = pd.DataFrame(results)
        
        # Sort by P_Value (Underscore matches now)
        df_results = df_results.sort_values(by='P_Value')
        
        # Save to CSV
        df_results.to_csv(output_file, index=False)
        
        print("\n--- ANALYSIS COMPLETE ---")
        print(f"Top 5 Significant Differences:")
        display(df_results.head(5))
        print(f"\nFull statistical table saved to: {os.path.abspath(output_file)}")

--- AUTOMATED STATISTICAL AUDIT ---
Data Loaded: 542 patients
Cohorts found: ['Public (Western)' 'Local (Bangladesh)']

Comparing: 'Public (Western)' vs. 'Local (Bangladesh)'
Analyzing 107 radiomic features...

--- ANALYSIS COMPLETE ---
Top 5 Significant Differences:


,Feature,Mean (Public (Western)),Mean (Local (Bangladesh)),Fold Change,P_Value,Test Used,Levene Variance P-Val,Significant?
93,original_shape_Sphericity,0.40 ± 0.05,0.72 ± 0.06,1.81x,2.766283e-62,Mann-Whitney U,8.008948e-02,YES
81,original_shape_Maximum3DDiameter,312.71 ± 52.93,141.95 ± 35.39,0.45x,1.370441e-60,Mann-Whitney U,8.579934e-02,YES
76,original_shape_Maximum2DDiameterColumn,305.92 ± 55.49,125.77 ± 30.28,0.41x,1.415556e-60,Mann-Whitney U,5.597968e-04,YES
75,original_shape_Maximum2DDiameterRow,291.82 ± 57.80,131.03 ± 32.33,0.45x,2.958347e-60,Mann-Whitney U,2.557274e-04,YES
106,original_shape_MajorAxisLength,249.63 ± 76.00,122.22 ± 31.34,0.49x,1.067975e-55,Mann-Whitney U,6.630252e-12,YES



Full statistical table saved to: d:\Thesis_Project\Results\table_statistical_analysis.csv
